## DEMO
1. Impact of the file layouts and row group size for filtering out rows during the scans. 
2. Is the default row group size (128MB) optimal in all the cases?
3. Check the combined impact of files and row groups skipping.

### Test data
[TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

![](doc/figures/tlc.png)

### Create a venv/kernel (optional)

In [ ]:
%%bash
PYTHON_VERSION=3.12.8
pyenv install $PYTHON_VERSION
pyenv local $PYTHON_VERSION
python -m venv venv
source venv/bin/activate
# optionally install Jupyer and/or create a Jupyter kernel

### PIP dependencies

In [ ]:
!pip install pyspark==3.5.2 sparkmeasure==0.25.0 pandas

### Download data (2024-2025)

In [ ]:
%%bash
#!/bin/bash

# Define the range of years and months you want to download
START_YEAR=2024
END_YEAR=2025

# Base URL for the Parquet files
BASE_URL="https://d37ci6vzurychx.cloudfront.net/trip-data"

# Create a directory to store the downloaded files
DOWNLOAD_DIR="../input/yellow_tripdata"
mkdir -p "$DOWNLOAD_DIR"

# Loop through each year and month
for YEAR in $(seq $START_YEAR $END_YEAR); do
    for MONTH in {1..12}; do
        # Format month with leading zero
        MONTH_PADDED=$(printf "%02d" $MONTH)
        # Construct the filename
        FILE_NAME="yellow_tripdata_${YEAR}-${MONTH_PADDED}.parquet"
        # Construct the full URL
        FILE_URL="${BASE_URL}/${FILE_NAME}"
        # Download the file
        echo "Downloading ${FILE_NAME}..."
        curl -o "${DOWNLOAD_DIR}/${FILE_NAME}" "$FILE_URL"
        # Check if the download was successful
        if [ $? -ne 0 ]; then
            echo "Failed to download ${FILE_NAME}"
        fi
    done
done


In [ ]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = f"--packages org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.9.0,ch.cern.sparkmeasure:spark-measure_2.12:0.25 \
--conf spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions \
--conf spark.sql.catalog.spark_catalog=org.apache.iceberg.spark.SparkSessionCatalog \
--conf spark.sql.catalog.spark_catalog.type=hive \
--conf spark.sql.catalogImplementation=hive \
--conf spark.sql.catalog.local=org.apache.iceberg.spark.SparkCatalog \
--conf spark.sql.catalog.local.type=hadoop \
--conf spark.sql.catalog.local.warehouse={os.getcwd()}/warehouse \
--conf spark.sql.defaultCatalog=local \
--conf spark.log.level=ERROR \
--driver-memory 8g pyspark-shell"

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder.master("local[1]").getOrCreate()

### Temporary view 

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW yellow_tripdata_temp_view
USING parquet
OPTIONS (
  path '{os.getcwd()}/../input/yellow_tripdata/yellow_tripdata_*.parquet'
);
""")

In [91]:
spark.sql("""SELECT count(*) FROM yellow_tripdata_temp_view""").show()

+--------+
|count(1)|
+--------+
|52367746|
+--------+



In [ ]:
### Dataset overview

In [92]:
# https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf
spark.sql("DESCRIBE yellow_tripdata_temp_view").show(100, False)

+---------------------+-------------+-------+
|col_name             |data_type    |comment|
+---------------------+-------------+-------+
|VendorID             |int          |NULL   |
|tpep_pickup_datetime |timestamp_ntz|NULL   |
|tpep_dropoff_datetime|timestamp_ntz|NULL   |
|passenger_count      |bigint       |NULL   |
|trip_distance        |double       |NULL   |
|RatecodeID           |bigint       |NULL   |
|store_and_fwd_flag   |string       |NULL   |
|PULocationID         |int          |NULL   |
|DOLocationID         |int          |NULL   |
|payment_type         |bigint       |NULL   |
|fare_amount          |double       |NULL   |
|extra                |double       |NULL   |
|mta_tax              |double       |NULL   |
|tip_amount           |double       |NULL   |
|tolls_amount         |double       |NULL   |
|improvement_surcharge|double       |NULL   |
|total_amount         |double       |NULL   |
|congestion_surcharge |double       |NULL   |
|Airport_fee          |double     

## Create tables with different layouts, file and row group sizes

In [74]:
test_tables = ["base","base_rg_512","base_rg_16", "sort", "sort_rg_16", "sort_rg_512" ,"zorder", "zorder_rg_16", "zorder_rg_512"]
test_tables = ["base_rg_512"]
default_target_file_size_bytes = 536870912 # 512MB
default_row_group_size_bytes = 134217728 # 128MB

optim_target_file_size_bytes = 16*1048576  # 16MB
optim_row_group_size_bytes = 4*1048576 # 4MB

q1 = """
    SELECT *
    FROM  demo.yellow_tripdata
    WHERE PULocationID = 138   -- e.g. JFK Airport zone
    AND DOLocationID = 236 ;
"""
q2= """
    SELECT *
    FROM  demo.yellow_tripdata
    WHERE PULocationID = 138   -- e.g. JFK Airport zone
    AND total_amount > 100 AND total_amount < 150 ;
"""
base_timing = None
sort_timing = None
sort_rg_timing = None
zorder_timing = None
zorder_rg_timing = None

In [75]:
for s in test_tables:
    spark.sql(f"DROP TABLE IF EXISTS demo.yellow_tripdata_{s}")

In [76]:
for s in test_tables:
    if s == "base":
        target_file_size_bytes = default_target_file_size_bytes
        row_group_size_bytes = default_row_group_size_bytes
    elif s == "base_rg_512":
        target_file_size_bytes = default_target_file_size_bytes
        row_group_size_bytes = optim_row_group_size_bytes
    elif s == "base_rg_16":
        target_file_size_bytes = optim_target_file_size_bytes
        row_group_size_bytes = optim_row_group_size_bytes
    elif s == "sort":
        target_file_size_bytes = default_target_file_size_bytes
        row_group_size_bytes = default_row_group_size_bytes
    elif s == "sort_rg_16":
        target_file_size_bytes = optim_target_file_size_bytes
        row_group_size_bytes = optim_row_group_size_bytes
    elif s =="sort_rg_512":
        target_file_size_bytes = default_target_file_size_bytes
        row_group_size_bytes = optim_row_group_size_bytes
    elif s == "zorder":
        target_file_size_bytes = default_target_file_size_bytes
        row_group_size_bytes = default_row_group_size_bytes
    elif s == "zorder_rg_16":
        target_file_size_bytes = optim_target_file_size_bytes
        row_group_size_bytes = optim_row_group_size_bytes
    elif s == "zorder_rg_512":
        target_file_size_bytes = default_target_file_size_bytes
        row_group_size_bytes = optim_row_group_size_bytes
    print(f"Creating table demo.yellow_tripdata_{s} with t:{target_file_size_bytes} bytes r:{row_group_size_bytes} bytes")
    spark.sql(f"""
     CREATE OR REPLACE TABLE demo.yellow_tripdata_{s}
        USING iceberg
        PARTITIONED BY (month(tpep_pickup_datetime)) 
        TBLPROPERTIES('write.target-file-size-bytes'='{target_file_size_bytes}', 'write.parquet.row-group-size-bytes'='{row_group_size_bytes}')
     AS 
     SELECT * FROM yellow_tripdata_temp_view 
     WHERE tpep_pickup_datetime >= '2024-01-01' AND tpep_pickup_datetime < '2025-04-01'""")

Creating table demo.yellow_tripdata_base_rg_512 with t:536870912 bytes r:4194304 bytes


### Data files rewriting - sort and z-order for different RG and file sizes

In [52]:
spark.sql("""
CALL local.system.rewrite_data_files(
  table      => 'demo.yellow_tripdata_sort',
  strategy => 'sort',
  sort_order => '(PULocationID ASC NULLS LAST, DOLocationID ASC NULLS LAST)',
  options => map('rewrite-all', 'true' )
)
""").toPandas()

,rewritten_data_files_count,added_data_files_count,rewritten_bytes_count,failed_data_files_count
0,15,15,823490305,0


In [56]:
spark.sql("""
CALL local.system.rewrite_data_files(
  table      => 'demo.yellow_tripdata_sort_rg_16',
  strategy => 'sort',
  sort_order => '(PULocationID ASC NULLS LAST, DOLocationID ASC NULLS LAST)',
  options => map('rewrite-all', 'true' )
)
""").toPandas()

,rewritten_data_files_count,added_data_files_count,rewritten_bytes_count,failed_data_files_count
0,62,56,860310771,0


In [55]:
spark.sql("""
CALL local.system.rewrite_data_files(
  table      => 'demo.yellow_tripdata_sort_rg_512',
  strategy => 'sort',
  sort_order => '(PULocationID ASC NULLS LAST, DOLocationID ASC NULLS LAST)',
  options => map('rewrite-all', 'true' )
)
""").toPandas()

,rewritten_data_files_count,added_data_files_count,rewritten_bytes_count,failed_data_files_count
0,15,15,860984335,0


In [24]:
spark.sql("""
CALL local.system.rewrite_data_files(
  table      => 'demo.yellow_tripdata_sort_rg_512',
  strategy => 'sort',
  sort_order => '(PULocationID ASC NULLS LAST, DOLocationID ASC NULLS LAST)',
  options => map('rewrite-all', 'true' )
)
""").toPandas()

,rewritten_data_files_count,added_data_files_count,rewritten_bytes_count,failed_data_files_count
0,15,15,860984335,0


In [59]:
spark.sql("""
CALL local.system.rewrite_data_files(
  table      => 'demo.yellow_tripdata_zorder',
  sort_order => 'zorder(PULocationID, DOLocationID)',
  options => map('rewrite-all', 'true' )
)
""").toPandas()

,rewritten_data_files_count,added_data_files_count,rewritten_bytes_count,failed_data_files_count
0,15,15,823490305,0


In [70]:
spark.sql("""
CALL local.system.rewrite_data_files(
  table      => 'demo.yellow_tripdata_zorder_rg_512',
  sort_order => 'zorder(PULocationID, DOLocationID)',
  options => map('rewrite-all', 'true' )
)
""").toPandas()

,rewritten_data_files_count,added_data_files_count,rewritten_bytes_count,failed_data_files_count
0,15,15,860984335,0


In [61]:
spark.sql("""
CALL local.system.rewrite_data_files(
  table      => 'demo.yellow_tripdata_zorder_rg_16',
  sort_order => 'zorder(PULocationID, DOLocationID)',
  options => map('rewrite-all', 'true' )
)
""").toPandas()

,rewritten_data_files_count,added_data_files_count,rewritten_bytes_count,failed_data_files_count
0,62,56,860310771,0


In [49]:
%%timeit
## base

r = spark.sql("""

    SELECT *
    FROM  demo.yellow_tripdata_base
    WHERE PULocationID = 138   -- e.g. JFK Airport zone
    AND DOLocationID = 236 ;
""").count()

876 ms ± 18.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [77]:
%%timeit
## base

r = spark.sql("""

    SELECT *
    FROM  demo.yellow_tripdata_base_rg_512
    WHERE PULocationID = 138   -- e.g. JFK Airport zone
    AND DOLocationID = 236 ;
""").count()

1.02 s ± 45.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [72]:
%%timeit
## base

r = spark.sql("""

    SELECT *
    FROM  demo.yellow_tripdata_base_rg_16
    WHERE PULocationID = 138   -- e.g. JFK Airport zone
    AND DOLocationID = 236 ;
""").count()

1.08 s ± 31.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [53]:
%%timeit
## sort

r = spark.sql("""

    SELECT *
    FROM  demo.yellow_tripdata_sort
    WHERE PULocationID = 138   -- e.g. JFK Airport zone
    AND DOLocationID = 236 ;
""").count()

470 ms ± 15.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [57]:
%%timeit
## sort_rg

r = spark.sql("""

    SELECT *
    FROM  demo.yellow_tripdata_sort_rg_512
    WHERE PULocationID = 138   -- e.g. JFK Airport zone
    AND DOLocationID = 236 ;
""").count()

209 ms ± 11 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [58]:
%%timeit
## sort_rg

r = spark.sql("""

    SELECT *
    FROM  demo.yellow_tripdata_sort_rg_16
    WHERE PULocationID = 138   -- e.g. JFK Airport zone
    AND DOLocationID = 236 ;
""").count()

102 ms ± 5.49 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [62]:
%%timeit
## z-order

r = spark.sql("""

    SELECT *
    FROM  demo.yellow_tripdata_zorder
    WHERE PULocationID = 138   -- e.g. JFK Airport zone
    AND DOLocationID = 236 ;
""").count()

434 ms ± 11.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [71]:
%%timeit
## z-order

r = spark.sql("""

    SELECT *
    FROM  demo.yellow_tripdata_zorder_rg_512
    WHERE PULocationID = 138   -- e.g. JFK Airport zone
    AND DOLocationID = 236 ;
""").count()

240 ms ± 8.37 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [66]:
%%timeit
## z-order

r = spark.sql("""

    SELECT *
    FROM  demo.yellow_tripdata_zorder_rg_16
    WHERE PULocationID = 138   -- e.g. JFK Airport zone
    AND DOLocationID = 236 ;
""").count()

201 ms ± 9.49 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [79]:
from sparkmeasure import StageMetrics
from sparkmeasure import TaskMetrics
stagemetrics = StageMetrics(spark)
taskmetrics = TaskMetrics(spark)

In [90]:
taskmetrics.runandmeasure(globals(),"""spark.sql("SELECT *FROM  demo.yellow_tripdata_base WHERE PULocationID = 138 AND DOLocationID = 236").count()
""")


Scheduling mode = FIFO
Spark Context default degree of parallelism = 1

Aggregated Spark task metrics:
numTasks => 16
successful tasks => 16
speculative tasks => 0
taskDuration => 969 (1.0 s)
schedulerDelayTime => 27 (27 ms)
executorRunTime => 941 (0.9 s)
executorCpuTime => 884 (0.9 s)
executorDeserializeTime => 0 (0 ms)
executorDeserializeCpuTime => 0 (0 ms)
resultSerializationTime => 1 (1 ms)
jvmGCTime => 7 (7 ms)
shuffleFetchWaitTime => 0 (0 ms)
shuffleWriteTime => 0 (0 ms)
gettingResultTime => 0 (0 ms)
resultSize => 4936 (4.8 KB)
diskBytesSpilled => 0 (0 Bytes)
memoryBytesSpilled => 0 (0 Bytes)
peakExecutionMemory => 0
recordsRead => 52367691
bytesRead => 94995102 (90.6 MB)
recordsWritten => 0
bytesWritten => 0 (0 Bytes)
shuffleRecordsRead => 15
shuffleTotalBlocksFetched => 15
shuffleLocalBlocksFetched => 15
shuffleRemoteBlocksFetched => 0
shuffleTotalBytesRead => 885 (885 Bytes)
shuffleLocalBytesRead => 885 (885 Bytes)
shuffleRemoteBytesRead => 0 (0 Bytes)
shuffleRemoteBytesReadT

In [89]:
taskmetrics.runandmeasure(globals(),"""spark.sql("SELECT *FROM  demo.yellow_tripdata_sort WHERE PULocationID = 138 AND DOLocationID = 236").count()
""")


Scheduling mode = FIFO
Spark Context default degree of parallelism = 1

Aggregated Spark task metrics:
numTasks => 16
successful tasks => 16
speculative tasks => 0
taskDuration => 393 (0.4 s)
schedulerDelayTime => 19 (19 ms)
executorRunTime => 374 (0.4 s)
executorCpuTime => 360 (0.4 s)
executorDeserializeTime => 0 (0 ms)
executorDeserializeCpuTime => 0 (0 ms)
resultSerializationTime => 0 (0 ms)
jvmGCTime => 0 (0 ms)
shuffleFetchWaitTime => 0 (0 ms)
shuffleWriteTime => 0 (0 ms)
gettingResultTime => 0 (0 ms)
resultSize => 4850 (4.7 KB)
diskBytesSpilled => 0 (0 Bytes)
memoryBytesSpilled => 0 (0 Bytes)
peakExecutionMemory => 0
recordsRead => 52367691
bytesRead => 2427964 (2.3 MB)
recordsWritten => 0
bytesWritten => 0 (0 Bytes)
shuffleRecordsRead => 15
shuffleTotalBlocksFetched => 15
shuffleLocalBlocksFetched => 15
shuffleRemoteBlocksFetched => 0
shuffleTotalBytesRead => 885 (885 Bytes)
shuffleLocalBytesRead => 885 (885 Bytes)
shuffleRemoteBytesRead => 0 (0 Bytes)
shuffleRemoteBytesReadToD

### Results
| Test case      | Read`*` [MB] | Files read | Files skipped | Output rows | Time [ms] | Stdev [ms] | Filter/Agg(WSCG) time [ms] |
|----------------|--------------|------------|---------------|-------------|-----------|------------|----------------------------|
| base-512       | 785,34       | 15         | 0             | 52,367,691  | 876       | 18.6       | 820                        |
| base-rg-512    | 807.07       | 15         | 0             | 52,367,691  | 1020      | 12.8       | 950                        |
| base-rg-16     | 820.46       | 62         | 0             | 52,367,691  | 1080      | 31.6       | 884                        |
| sort-512       | 824.82       | 15         | 0             | 52,367,691  | 470       | 11.1       | 408                        |
| sort-rg-512    | 807.07       | 15         | 0             | 3,991,092   | 209       | 11         | 57                         |
| sort-rg-16     | **249.72**   | 16         | 39            | 4,162,783   | **102**   | 5.49       | 43                         |
| z-order-512    | 819.10       | 15         | 0             | 52,367,691  | 434       | 11.1       | 387                        |
| z-order-rg-512 | 810.76       | 15         | 0             | 8,699,663   | 240       | 8.37       | 108                        |
| z-order-rg-16  | 654.68       | 45         | 11            | 8,830,018   | 201       | 9.49       | 87                         |

`*` Depends on files read/skip.

1. Reduction of the number of output rows as result of the file rewrite but only if we adjust properly the `Row group` size to the actual size of our files(compare `base` vs `rg` 128MB -> 4MB for sort and z-order).
2. Best results if you can combine files and row groups skipping (compare `rg-512` -> `rg-16`).
3. Even if we don't have a reduction of output rows - the subsequent filter phase can benefit a lot from pre-sorted data (compare `Filter time` for the same number of output rows base vs sort/z-order but 90.6 MB vs 2.3MB)
4. Time reduction **~9x**, bytes read **~3x** (base vs sort-rg-16), ~4.5x / 20% (base vs z-order-rg-16)



In [14]:
%%timeit  -v base_timing 
## base

r = spark.sql("""

    SELECT *
    FROM  demo.yellow_tripdata_base
    WHERE PULocationID = 138   -- e.g. JFK Airport zone
    AND total_amount > 100 AND total_amount < 150 ;
""").count()

659 ms ± 9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [15]:
%%timeit
## base

r = spark.sql("""

    SELECT *
    FROM  demo.yellow_tripdata_base_rg
    WHERE PULocationID = 138   -- e.g. JFK Airport zone
    AND total_amount > 100 AND total_amount < 150 ;
""").count()

1.08 s ± 20.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [16]:
spark.sql("""
CALL local.system.rewrite_data_files(
  table      => 'demo.yellow_tripdata_zorder_rg',
  sort_order => 'zorder(PULocationID, total_amount)',
  options => map('rewrite-all', 'true' )
)
""").toPandas()

,rewritten_data_files_count,added_data_files_count,rewritten_bytes_count,failed_data_files_count
0,56,46,751880957,0


In [20]:
%%timeit

r = spark.sql("""

    SELECT *
    FROM  demo.yellow_tripdata_zorder_rg
    WHERE PULocationID = 138   -- e.g. JFK Airport zone
    AND total_amount > 100 AND total_amount < 150 ;
""").count()

98 ms ± 2.63 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
spark.sql("""
CALL local.system.rewrite_data_files(
  table      => 'demo.yellow_tripdata_sort_rg',
  strategy => 'sort',
  sort_order => '(PULocationID ASC NULLS LAST, total_amount ASC NULLS LAST)',
  options => map('rewrite-all', 'true' )
)
""").toPandas()

In [19]:
%%timeit

r = spark.sql("""

    SELECT *
    FROM  demo.yellow_tripdata_sort_rg
    WHERE PULocationID = 138   -- e.g. JFK Airport zone
    AND total_amount > 100 AND total_amount < 150 ;
""").count()

122 ms ± 1.58 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


### Results
| Test case      | Read [MB]  | Files read | Files skipped | Output rows | Time [ms] | Stdev [ms] |
|----------------|------------|------------|---------------|-------------|-----------|------------|
| base-512       | 785,34     | 15         | 0             | 52,367,691  | 730       | 22.2       |
| z-order-rg-16  | **207,21** | 15         | 41            | 2,124,073   | *98*      | 2.63       |
| sort-rg-16     | 232,88     | 17         | 39            | 4,007,269   | 122       | 1.58       |

1. Time reduction  **~7.5x** / bytes read **~3.8x** (base vs z-order-rg-16) , ~6x/~3.4x (base vs sort-rg-16)

In [ ]:
spark.stop()